# Ch 16 · Collision II: Geom Types and Primitive Colliders

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/16-primitive-narrowphase.ipynb)

Companion notebook for the chapter page `chapters/16-primitive-narrowphase.html` ([page](../chapters/16-primitive-narrowphase.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## 1. Geom types and the dispatch table
`collision_driver.py::MJ_COLLISION_TABLE` maps every ordered type pair `(t1 <= t2)` to PRIMITIVE, CONVEX or nothing. Example 16.1 prints it from the library's own dicts, plus `MJ_FLEX_COLLISION_TABLE` and the one cell `DisableBit.NATIVECCD` flips (box-box).

In [ ]:
# Example 16.1 - ex1_collision_table.py
"""Example 16.1 - the collision dispatch table, read from MJWarp itself.

Prints MJ_COLLISION_TABLE as a GeomType x GeomType grid (P = PRIMITIVE, C = CONVEX, . = never
collides), the function that handles each cell, the flex table, and what DisableBit.NATIVECCD
does to box-box. Nothing here is hand-typed: every cell comes from the library's own dicts.
Run: python ex1_collision_table.py [--dump-json PATH]
  --dump-json PATH writes the grid (both NATIVECCD settings + flex row) for the page's heatmap.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.collision_driver import MJ_COLLISION_TABLE      # private API
from mujoco_warp._src.collision_driver import MJ_FLEX_COLLISION_TABLE  # private API
from mujoco_warp._src.collision_primitive import _PRIMITIVE_COLLISIONS  # private API

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

G = mjw.GeomType
RIGID = [G.PLANE, G.HFIELD, G.SPHERE, G.CAPSULE, G.ELLIPSOID, G.CYLINDER, G.BOX, G.MESH]
SHORT = {G.PLANE: "plane", G.HFIELD: "hfield", G.SPHERE: "sphere", G.CAPSULE: "capsule",
         G.ELLIPSOID: "ellips", G.CYLINDER: "cylind", G.BOX: "box", G.MESH: "mesh",
         G.FLEX: "flex"}


def table_for(nativeccd_disabled):
    """Same logic as collision_driver._narrowphase: copy the table, flip BOX-BOX if disabled."""
    table = dict(MJ_COLLISION_TABLE)
    if nativeccd_disabled:
        table[(G.BOX, G.BOX)] = type(table[(G.BOX, G.BOX)]).PRIMITIVE
    return table


def handler(t1, t2, kind):
    if kind == "PRIMITIVE":
        return _PRIMITIVE_COLLISIONS[(t1, t2)].key          # the wp.func name
    if G.HFIELD in (t1, t2):
        return f"ccd_hfield_kernel_builder({t1.name}, {t2.name})"
    return f"ccd_kernel_builder({t1.name}, {t2.name})"


def grid(table):
    cells = {}
    for i, t1 in enumerate(RIGID):
        for t2 in RIGID[i:]:
            v = table.get((t1, t2))
            cells[(t1, t2)] = None if v is None else v.name
    return cells


def show(cells, title):
    print(title)
    print("         " + " ".join(f"{SHORT[t]:>7s}" for t in RIGID))
    for i, t1 in enumerate(RIGID):
        row = []
        for j, t2 in enumerate(RIGID):
            a, b = (t1, t2) if i <= j else (t2, t1)
            k = cells[(a, b)]
            row.append(f"{'.' if k is None else k[0]:>7s}")
        print(f"{SHORT[t1]:>8s} " + " ".join(row))


default = grid(table_for(False))
noccd = grid(table_for(True))
show(default, "MJ_COLLISION_TABLE (P = PRIMITIVE, C = CONVEX, . = no entry: never collides)")
n_p = sum(v == "PRIMITIVE" for v in default.values())
n_c = sum(v == "CONVEX" for v in default.values())
n_none = sum(v is None for v in default.values())
print(f"cells: {n_p} PRIMITIVE, {n_c} CONVEX, {n_none} missing ->",
      [f"{a.name}-{b.name}" for (a, b), v in default.items() if v is None])

print("\nprimitive handlers (collision_primitive._PRIMITIVE_COLLISIONS):")
for (t1, t2), f in _PRIMITIVE_COLLISIONS.items():
    tag = "" if default[(t1, t2)] == "PRIMITIVE" else "   <- only with NATIVECCD disabled"
    print(f"  {t1.name:>8s}-{t2.name:<9s} {f.key}{tag}")

print("\nBOX-BOX by default:", default[(G.BOX, G.BOX)],
      "| with DisableBit.NATIVECCD:", noccd[(G.BOX, G.BOX)])
flipped = [f"{a.name}-{b.name}" for (a, b) in default if default[(a, b)] != noccd[(a, b)]]
print("cells changed by NATIVECCD:", flipped)

print("\nMJ_FLEX_COLLISION_TABLE:")
for (t1, t2), v in MJ_FLEX_COLLISION_TABLE.items():
    print(f"  {t1.name:>9s}-{t2.name:<5s} {v.name}")
missing_flex = [t.name for t in RIGID if (t, G.FLEX) not in MJ_FLEX_COLLISION_TABLE]
print("flex partners with no entry:", missing_flex)
print("SDF in the rigid table:", any(G.SDF in k for k in MJ_COLLISION_TABLE),
      "(SDF pairs go to sdf_narrowphase instead)")

if args.dump_json:
    out = {"types": [SHORT[t] for t in RIGID], "cells": []}
    for (t1, t2), kind in default.items():
        out["cells"].append({"a": RIGID.index(t1), "b": RIGID.index(t2), "kind": kind,
                             "kind_noccd": noccd[(t1, t2)],
                             "fn": None if kind is None else handler(t1, t2, kind),
                             "fn_noccd": None if noccd[(t1, t2)] is None
                             else handler(t1, t2, noccd[(t1, t2)])})
    out["flex"] = [{"a": RIGID.index(t1), "kind": v.name} for (t1, _), v
                   in MJ_FLEX_COLLISION_TABLE.items()]
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(out["cells"]), "cells")

## 2-4. Pair order, the Geom struct and the launches
`_add_geom_pair` stores the lower geom type first, so `contact.frame[0]` points from the lower type to the higher. The primitive path is **one** `wp.launch(_primitive_narrowphase(...), dim=d.naconmax)` whose static if-chain holds only the pair types with `m.geom_pair_type_count > 0`; the convex path launches one kernel per present convex pair type (a grid-stride loop for non-heightfield pairs; heightfield pairs get one thread per pool slot). Example 16.8 records the launches by wrapping `wp.launch` (module hashes can differ on your machine; on GPU the convex `dim` is a few device waves).

In [ ]:
# Example 16.8 - ex8_type_pair_launches.py
"""Example 16.8 - what the narrowphase actually launches, and what changes when a geom type goes.

We wrap wp.launch to record every kernel launched by mjw.collision (name and dim), and wrap
collision_driver.create_collision_context to keep the broadphase output, so we can say which
static branch each primitive thread takes. Scene A has PLANE, SPHERE, BOX and CYLINDER geoms
(one convex pair type: BOX-CYLINDER). Scene B is the same without the spheres. Expect:
  - exactly ONE primitive launch in both scenes, dim = d.naconmax, but a different compiled
    kernel (its if-chain is built from the pair types with geom_pair_type_count > 0);
  - one convex launch per present convex pair type (here BOX-CYLINDER), unchanged.
Run: python ex8_type_pair_launches.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import collision_driver           # private API
from mujoco_warp._src import warp_util                  # private API (cache_kernel's dict)
from mujoco_warp._src.collision_driver import MJ_COLLISION_TABLE  # private API
from mujoco_warp._src.math import upper_trid_index      # private API

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

SCENE = """
<mujoco model="dispatch">
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="crate" pos="0 0 0.0995"><freejoint/>
      <geom name="crate" type="box" size="0.1 0.1 0.1"/></body>
    <body name="drum" pos="0.1595 0 0.0995"><freejoint/>
      <geom name="drum" type="cylinder" size="0.06 0.1"/></body>
    {spheres}
  </worldbody>
</mujoco>
"""
SPHERES = """
    <body name="ball_top" pos="0 0 0.249"><freejoint/>
      <geom name="ball_top" type="sphere" size="0.05"/></body>
    <body name="ball_floor" pos="-0.3 0 0.0495"><freejoint/>
      <geom name="ball_floor" type="sphere" size="0.05"/></body>"""

G = mjw.GeomType
NWORLD = 2
_launch = wp.launch
_make_ctx = collision_driver.create_collision_context
calls, ctxs = [], []


def recording_launch(kernel, dim, *a, **kw):
    calls.append((kernel.key, kernel.module.name, dim))
    return _launch(kernel, dim, *a, **kw)


def recording_ctx(naconmax):
    ctx = _make_ctx(naconmax)
    ctxs.append(ctx)
    return ctx


def present_pairs(m):
    rigid = [t for t in G if t.value <= G.MESH.value]       # geom types a geom can have here
    out = []
    for t1 in rigid:
        for t2 in rigid:
            if t1.value <= t2.value:
                if m.geom_pair_type_count[upper_trid_index(len(G), t1.value, t2.value)]:
                    out.append(f"{t1.name}-{t2.name}")
    return out


report = {}
for label, spheres in [("A: plane+sphere+box+cylinder", SPHERES), ("B: spheres removed", "")]:
    mjm = mujoco.MjModel.from_xml_string(SCENE.format(spheres=spheres))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=24, njmax=96)
    mjw.kinematics(m, d)
    mjw.collision(m, d)                     # warm-up: compile outside the recording
    calls.clear(), ctxs.clear()
    wp.launch = recording_launch
    collision_driver.create_collision_context = recording_ctx
    try:
        mjw.collision(m, d)
    finally:
        wp.launch = _launch
        collision_driver.create_collision_context = _make_ctx
    wp.synchronize()

    print(label)
    print("  geom_pair_type_count > 0:", present_pairs(m))
    for key, module, dim in calls:
        print(f"  launch {key.split('__')[0]:22s} module {module.split('_')[-1]:>9s}  dim = {dim}")
    ncol = min(int(d.ncollision.numpy()[0]), d.naconmax)
    nacon = min(int(d.nacon.numpy()[0]), d.naconmax)
    pair = ctxs[0].collision_pair.numpy()[:ncol]
    wid = ctxs[0].collision_worldid.numpy()[:ncol]
    types = [(G(int(mjm.geom_type[a])), G(int(mjm.geom_type[b]))) for a, b in pair]
    branch = [f"{a.name}-{b.name}" if MJ_COLLISION_TABLE[(a, b)].name == "PRIMITIVE"
              else f"none ({a.name}-{b.name} is convex)" for a, b in types]
    print(f"  d.naconmax = {d.naconmax}, d.ncollision = {ncol}, d.nacon = {nacon}")
    print("  primitive kernel, thread -> world, static branch taken:")
    for i in range(ncol):
        print(f"    tid {i:2d}  w{wid[i]}  {branch[i]}")
    print(f"    tid {ncol:2d} .. {d.naconmax - 1}: tid >= ncollision, return at once")
    cg = d.contact.geom.numpy()[:nacon]
    cw = d.contact.worldid.numpy()[:nacon]
    ctype = [f"{G(int(mjm.geom_type[a])).name}-{G(int(mjm.geom_type[b])).name}" for a, b in cg]
    kinds, counts = np.unique(ctype, return_counts=True)
    print("  contacts per type pair (both worlds):", dict(zip(kinds.tolist(), counts.tolist())))
    report[label] = {"pairs": present_pairs(m),
                     "launches": [[k.split("__")[0], mod.split("_")[-1], np.atleast_1d(n).tolist()]
                                  for k, mod, n in calls],
                     "naconmax": int(d.naconmax), "ncollision": ncol, "nacon": nacon,
                     "threads": [[int(wid[i]), branch[i]] for i in range(ncol)],
                     "contacts": [[int(cw[i]), ctype[i]] for i in range(nacon)]}

prim = [k for k in warp_util._KERNEL_CACHE if k[-1] == "_primitive_narrowphase"]
print("compiled _primitive_narrowphase variants in the cache_kernel dict:", len(prim))

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(report, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 5. `write_contact`: detected, active, margin and gap
Detected: `dist < margin + gap` (strict, float32). Active: `dist < margin`, or in the gap band with adhesion (then condim 1). Note the boundary row at exactly `margin + gap`.

In [ ]:
# Example 16.6 - ex6_margin_gap.py
"""Example 16.6 - margin and gap: detected vs active contacts, and the strict-< boundary.

write_contact (collision_core.py) decides, per contact:
    detected = dist < margin + gap                     -> a slot in d.contact is used
    active   = dist < margin  (or, with adhesion != 0, dist < margin + gap)
make_constraint then builds constraint rows only for active contacts (dist < includemargin, with
includemargin = margin). A sphere (margin 0.02, gap 0.03) hovers above a floor at a different
height in each world; we compare MJWarp with MuJoCo C (mj_forward) world by world, then repeat
with adhesion so in-gap contacts become active with condim 1.
Run: python ex6_margin_gap.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

HOVER = """
<mujoco model="margin_gap">
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size="0.1" margin="0.02" gap="0.03" adhesion="{adh}"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

GAPS = np.array([-0.005, 0.01, 0.019, 0.021, 0.04, 0.049, 0.05, 0.051, 0.06])  # surface gap
NWORLD = len(GAPS)
rows = {}

for adh in [0.0, 0.5]:
    mjm = mujoco.MjModel.from_xml_string(HOVER.format(adh=adh))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
    qpos = np.tile(mjm.qpos0, (NWORLD, 1))
    qpos[:, 2] = 0.1 + GAPS
    d.qpos.assign(qpos.astype(np.float32))
    mjw.forward(m, d)                        # kinematics -> collision -> constraints -> solve

    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    wid = d.contact.worldid.numpy()[:n]
    cdist = dict(zip(wid, d.contact.dist.numpy()[:n]))
    cdim = dict(zip(wid, d.contact.dim.numpy()[:n]))
    incl = d.contact.includemargin.numpy()[:n]
    nefc = d.nefc.numpy()

    mjd = mujoco.MjData(mjm)
    print(f"adhesion = {adh}   (margin 0.02, gap 0.03, includemargin = {incl[0] if n else '-'})")
    print("  gap      MJWarp: dist      detected dim nefc | C: ncon nefc")
    out = []
    for w, g in enumerate(GAPS):
        mjd.qpos[:] = qpos[w]
        mujoco.mj_forward(mjm, mjd)
        det = w in cdist
        dstr = f"{cdist[w]: .6f}" if det else "    -    "
        dim = cdim[w] if det else "-"
        print(f"  {g:6.3f}   {dstr}   {str(det):5s}   {dim}  {nefc[w]:3d}  |"
              f"    {mjd.ncon}   {mjd.nefc:3d}")
        out.append({"gap": float(g), "detected": bool(det), "nefc": int(nefc[w]),
                    "c_ncon": int(mjd.ncon), "c_nefc": int(mjd.nefc)})
    rows[str(adh)] = out
    print("  overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

print("boundary row (gap 0.05): MJWarp keeps dist < margin + gap (strict, float32); C uses float64")
print(f"  float32 (MJWarp): dist = {np.float32(0.15) - np.float32(0.1):.9f}"
      f"  vs  margin + gap = {np.float32(0.02) + np.float32(0.03):.9f}  -> dropped")
print(f"  float64 (C):      dist = {0.15 - 0.1:.17f}  vs  margin + gap = {0.02 + 0.03:.17f}"
      "  -> kept")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"margin": 0.02, "gap": 0.03, "rows": rows}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. Material mixing
`contact_material_params`: explicit pair first, then priority (in 4.0.1 a direct, negative solref on either geom still gives the element-wise min), else max condim, element-wise max friction, solmix-weighted solref/solimp and summed adhesion; friction is clamped to `mjMINMU`.

In [ ]:
# Example 16.5 - ex5_material_mixing.py
"""Example 16.5 - material mixing: how two geoms combine friction, condim, solref, solimp, adhesion.

Six balls with different materials rest on one floor. For each contact we print what MJWarp
wrote (contact.friction, dim, solref, solimp, adhesion) next to a NumPy re-implementation of
collision_core.contact_material_params:
  - higher geom_priority wins: its condim, friction, solimp, adhesion, and its solref if both
    solrefs are standard (timeconst > 0). In 4.0.1 a direct (negative) solref on either geom
    gives the element-wise min even with unequal priority (MuJoCo C uses the winner's solref);
  - equal priority: condim = max, friction = element-wise max, adhesion = a1 + a2,
    mix = solmix1 / (solmix1 + solmix2); solimp = mix * solimp1 + (1 - mix) * solimp2;
    solref is mixed the same way if both are standard (timeconst > 0), else element-wise min;
  - every friction value is clamped to at least mujoco.mjMINMU.
Run: python ex5_material_mixing.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=5, suppress=True)

MATERIALS = """
<mujoco model="materials">
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1" friction="1 0.005 0.0001"/>
    <body name="A" pos="0 0 0.099"><freejoint/>
      <geom name="A_plain" type="sphere" size="0.1" friction="0.5 0.02 0.0002"/></body>
    <body name="B" pos="0.5 0 0.099"><freejoint/>
      <geom name="B_priority" type="sphere" size="0.1" priority="1" friction="0.3 0 0"
            condim="1" solref="0.04 0.5"/></body>
    <body name="C" pos="1 0 0.099"><freejoint/>
      <geom name="C_solmix3" type="sphere" size="0.1" solmix="3" solref="0.05 1"
            solimp="0.8 0.9 0.01 0.5 2" condim="6"/></body>
    <body name="D" pos="1.5 0 0.099"><freejoint/>
      <geom name="D_direct" type="sphere" size="0.1" solref="-1000 -50"/></body>
    <body name="E" pos="2 0 0.099"><freejoint/>
      <geom name="E_sticky" type="sphere" size="0.1" adhesion="0.3"/></body>
    <body name="F" pos="2.5 0 0.099"><freejoint/>
      <geom name="F_lowprio_direct" type="sphere" size="0.1" priority="-1" solref="-1000 -50"/></body>
  </worldbody>
</mujoco>
"""


def mix_rule(mjm, g1, g2):
    """NumPy copy of contact_material_params (geom-pair branch, no explicit <pair>)."""
    p1, p2 = mjm.geom_priority[g1], mjm.geom_priority[g2]
    f1, f2 = mjm.geom_friction[g1], mjm.geom_friction[g2]
    s1, s2 = mjm.geom_solmix[g1], mjm.geom_solmix[g2]
    if p1 > p2:
        mix, condim, fr, adh = 1.0, mjm.geom_condim[g1], f1, mjm.geom_adhesion[g1]
    elif p2 > p1:
        mix, condim, fr, adh = 0.0, mjm.geom_condim[g2], f2, mjm.geom_adhesion[g2]
    else:
        tiny = mujoco.mjMINVAL
        if s1 < tiny and s2 < tiny:
            mix = 0.5
        elif s1 < tiny:
            mix = 0.0
        elif s2 < tiny:
            mix = 1.0
        else:
            mix = s1 / (s1 + s2)
        condim = max(mjm.geom_condim[g1], mjm.geom_condim[g2])
        fr = np.maximum(f1, f2)
        adh = mjm.geom_adhesion[g1] + mjm.geom_adhesion[g2]
    friction = np.maximum(mujoco.mjMINMU, [fr[0], fr[0], fr[1], fr[2], fr[2]])
    r1, r2 = mjm.geom_solref[g1], mjm.geom_solref[g2]
    solref = mix * r1 + (1 - mix) * r2 if (r1[0] > 0 and r2[0] > 0) else np.minimum(r1, r2)
    solimp = mix * mjm.geom_solimp[g1] + (1 - mix) * mjm.geom_solimp[g2]
    return {"dim": int(condim), "friction": friction, "solref": solref, "solimp": solimp,
            "adhesion": adh, "mix": mix}


mjm = mujoco.MjModel.from_xml_string(MATERIALS)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=16, njmax=64)
mjw.kinematics(m, d)
mjw.collision(m, d)

n = min(int(d.nacon.numpy()[0]), d.naconmax)
geom = d.contact.geom.numpy()[:n]
got = {"dim": d.contact.dim.numpy()[:n], "friction": d.contact.friction.numpy()[:n],
       "solref": d.contact.solref.numpy()[:n], "solimp": d.contact.solimp.numpy()[:n],
       "adhesion": d.contact.adhesion.numpy()[:n]}
name = lambda g: mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, int(g))

worst = 0.0
for i in np.argsort(geom[:, 1]):
    g1, g2 = geom[i]
    ref = mix_rule(mjm, g1, g2)
    print(f"{name(g1)} + {name(g2)}   (mix = {ref['mix']:.2f})")
    for k in ["dim", "friction", "solref", "solimp", "adhesion"]:
        err = float(np.max(np.abs(np.asarray(got[k][i], float) - np.asarray(ref[k], float))))
        worst = max(worst, err)
        a, b = np.asarray(got[k][i], float).round(6), np.asarray(ref[k], float).round(6)
        if k == "dim":
            a, b = int(a), int(b)
        print(f"   {k:9s} MJWarp {a}   rule {b}")
print(f"largest |MJWarp - rule| over all fields: {worst:.2e}")
print("mjMINMU =", mujoco.mjMINMU, "(floor for every friction coefficient)")
# Ball F: the floor wins on priority, but F's solref is direct. 4.0.1 takes the element-wise min;
# MuJoCo C (mj_contactParam) uses the winner's (floor's) solref.
mjd = mujoco.MjData(mjm)
mujoco.mj_forward(mjm, mjd)
gF = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_GEOM, "F_lowprio_direct")
iF = [i for i in range(n) if gF in geom[i]][0]
cF = [c for c in mjd.contact[:mjd.ncon] if gF in c.geom][0]
print(f"F solref: MJWarp 4.0.1 {got['solref'][iF].round(4)}   MuJoCo C {cF.solref.round(4)}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 7. Primitive colliders
### 7.1 Plane-sphere
`dist = (c - p)·n - r`, `pos = c - n (r + dist/2)`: the midpoint between the surfaces.

In [ ]:
# Example 16.2 - ex2_plane_sphere.py
"""Example 16.2 - plane-sphere: contact.dist and contact.pos against the closed-form formula.

A sphere of radius r sits above a tilted plane at a different height in every world. The
primitive collider plane_sphere computes
    dist = dot(c - p, n) - r          (signed gap between the surfaces, negative = overlap)
    pos  = c - n * (r + dist / 2)     (the MIDPOINT between the two surfaces)
We recompute both in NumPy and compare with what MJWarp wrote into d.contact.
Run: python ex2_plane_sphere.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLANE_SPHERE = """
<mujoco model="plane_sphere">
  <worldbody>
    <geom name="ramp" type="plane" size="5 5 .1" zaxis="0.3 0 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size="0.1" margin="0.2"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

HEIGHTS = np.array([0.05, 0.08, 0.10, 0.12, 0.16, 0.25], dtype=np.float32)  # sphere centre z
NWORLD = len(HEIGHTS)                    # one world per height (same on CPU and GPU)

mjm = mujoco.MjModel.from_xml_string(PLANE_SPHERE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)

qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qpos[:, 2] = HEIGHTS
d.qpos.assign(qpos)
mjw.kinematics(m, d)                     # geom_xpos / geom_xmat for every world
mjw.collision(m, d)                      # broadphase + narrowphase

n = min(int(d.nacon.numpy()[0]), d.naconmax)
wid = d.contact.worldid.numpy()[:n]
dist = d.contact.dist.numpy()[:n]
pos = d.contact.pos.numpy()[:n]
normal = d.contact.frame.numpy()[:n, 0]
order = np.argsort(wid)                  # pool order is arbitrary on a GPU; sort by world

n_plane = np.array([0.3, 0.0, 1.0]) / np.linalg.norm([0.3, 0.0, 1.0])
r = 0.1
print(f"plane normal n = {n_plane}, sphere radius r = {r}, margin = 0.2")
print("world  centre_z   dist(MJWarp)  dist(formula)  pos(MJWarp)                 |pos err|")
max_err = 0.0
rows = []
for i in order:
    c = np.array([0.0, 0.0, HEIGHTS[wid[i]]])
    dist_ref = c @ n_plane - r
    pos_ref = c - n_plane * (r + 0.5 * dist_ref)
    err = max(abs(dist[i] - dist_ref), np.abs(pos[i] - pos_ref).max())
    max_err = max(max_err, err)
    rows.append({"z": float(HEIGHTS[wid[i]]), "dist": round(float(dist[i]), 5),
                 "pos": [round(float(v), 5) for v in pos[i]]})
    print(f"  w{wid[i]}   {HEIGHTS[wid[i]]:.3f}     {dist[i]: .5f}      {dist_ref: .5f}     "
          f"{pos[i]}   {np.abs(pos[i] - pos_ref).max():.1e}")
print("normal of every contact (frame[0]) equals the plane normal:",
      bool(np.allclose(normal, n_plane, atol=1e-6)))
print(f"max |MJWarp - formula| = {max_err:.2e}  (float32)")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"normal": [round(float(v), 5) for v in n_plane], "r": r, "rows": rows}, f,
                  separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Interactive geometry: plane_sphere and capsule-capsule closest points, the same formulas as the page widgets.
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def plane_sphere(tilt_deg=10, cx=0.05, cz=0.12, r=0.08):
    th = np.radians(tilt_deg)
    n = np.array([-np.sin(th), np.cos(th)])          # plane normal through the origin
    c = np.array([cx, cz])
    dist = c @ n - r                                  # collision_primitive_core.plane_sphere
    pos = c - n * (r + 0.5 * dist)
    t = np.array([n[1], -n[0]])
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(*np.array([-t * 0.5, t * 0.5]).T, lw=3, label="plane")
    ax.add_patch(plt.Circle(c, r, alpha=0.3))
    ax.plot(*pos, "o", color="C3", label=f"contact.pos, dist = {dist:.4f}")
    ax.arrow(*pos, *(n * 0.06), width=0.003, color="C3")
    ax.set_aspect("equal"); ax.set_xlim(-0.4, 0.4); ax.set_ylim(-0.1, 0.3); ax.legend(loc="upper left")
    plt.show()

try:
    from ipywidgets import interact
    interact(plane_sphere, tilt_deg=(-30, 30, 1), cx=(-0.3, 0.3, 0.005), cz=(-0.02, 0.27, 0.005), r=(0.03, 0.12, 0.005))
except ImportError:
    plane_sphere()

### 7.3 Capsule-capsule
`_segment_weights` returns two closest-point pairs only for parallel segments (relative test `|ab x cd|^2 <= 1e-8 |ab|^2 |cd|^2`).

In [ ]:
# Example 16.3 - ex3_capsule_capsule.py
"""Example 16.3 - capsule-capsule: parallel axes give 2 contacts, crossed axes give 1.

A capsule is a segment plus a radius, so capsule-capsule reduces to "closest points between two
segments" (_segment_weights, a port of MuJoCo's mjc_segWeights) followed by sphere_sphere at
those points. For parallel segments the closest points are not unique: the collider returns the
two ends of the overlap, so the pair writes 2 contacts (geomcollisionid 0 and 1).
Run: python ex3_capsule_capsule.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

CAPSULES = """
<mujoco model="capsule_capsule">
  <worldbody>
    <geom name="rail" type="capsule" fromto="-0.3 0 0.5  0.3 0 0.5" size="0.05"/>
    <body name="rod" pos="0 0 0.59">
      <freejoint/>
      <geom name="rod" type="capsule" fromto="-0.2 0 0  0.2 0 0" size="0.05"/>
    </body>
  </worldbody>
</mujoco>
"""


def quat_z(deg):
    a = np.radians(deg) / 2
    return [np.cos(a), 0.0, 0.0, np.sin(a)]


# world: (description, x offset of the rod, yaw of the rod in degrees)
CASES = [("parallel, centred", 0.0, 0.0),
         ("parallel, shifted +0.3", 0.3, 0.0),
         ("yaw 30 deg", 0.0, 30.0),
         ("crossed (yaw 90 deg)", 0.0, 90.0)]
NWORLD = len(CASES)

mjm = mujoco.MjModel.from_xml_string(CAPSULES)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)

qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
for w, (_, dx, yaw) in enumerate(CASES):
    qpos[w, 0] = dx
    qpos[w, 3:7] = quat_z(yaw)
d.qpos.assign(qpos)
mjw.kinematics(m, d)
mjw.collision(m, d)

n = min(int(d.nacon.numpy()[0]), d.naconmax)
wid = d.contact.worldid.numpy()[:n]
gcid = d.contact.geomcollisionid.numpy()[:n]
dist = d.contact.dist.numpy()[:n]
pos = d.contact.pos.numpy()[:n]
normal = d.contact.frame.numpy()[:n, 0]

print("surfaces overlap by 0.01 m (centre distance 0.09, radii 0.05 + 0.05)")
for w, (name, _, _) in enumerate(CASES):
    sel = np.where(wid == w)[0]
    sel = sel[np.argsort(gcid[sel])]
    print(f"w{w} {name:24s} contacts: {len(sel)}")
    for i in sel:
        print(f"     gcid {gcid[i]}  dist {dist[i]: .4f}  pos {pos[i]}  normal {normal[i]}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

Sweep: one world per yaw angle of the rod; only parallel (to within the 1e-8 test, about 0.006 deg) gives 2 contacts.

In [ ]:
# Sweep the rod's yaw over one world per angle and count capsule-capsule contacts per world.
SWEEP = """
<mujoco><worldbody>
  <geom type="capsule" fromto="-0.3 0 0.5  0.3 0 0.5" size="0.05"/>
  <body pos="0 0 0.59"><freejoint/><geom type="capsule" fromto="-0.2 0 0  0.2 0 0" size="0.05"/></body>
</worldbody></mujoco>"""
yaw = np.linspace(0.0, 90.0, 91 if ON_GPU else 31)
mjm_s = mujoco.MjModel.from_xml_string(SWEEP)
m_s = mjw.put_model(mjm_s)
d_s = mjw.make_data(mjm_s, nworld=len(yaw), nconmax=4, njmax=16)
q = np.tile(mjm_s.qpos0, (len(yaw), 1))
q[:, 3], q[:, 6] = np.cos(np.radians(yaw) / 2), np.sin(np.radians(yaw) / 2)
d_s.qpos.assign(q.astype(np.float32))
mjw.kinematics(m_s, d_s)
mjw.collision(m_s, d_s)
n = min(int(d_s.nacon.numpy()[0]), d_s.naconmax)
count = np.bincount(d_s.contact.worldid.numpy()[:n], minlength=len(yaw))
print(f"contacts at yaw {yaw[:3].round(1).tolist()} deg:", count[:3], "| worlds with 2 contacts:", int((count == 2).sum()))
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 2.8))
    ax.step(yaw, count, where="mid")
    ax.set_xlabel("yaw of the rod [deg]"); ax.set_ylabel("contacts"); ax.set_yticks([0, 1, 2])
    ax.set_title("capsule-capsule: 2 contacts only when parallel")
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

### 7.5 Plane-box
All 8 corners go to `write_contact` with `id_ = i`; only those within margin + gap survive. The corner index becomes `contact.geomcollisionid`.

In [ ]:
# Example 16.4 - ex4_plane_box.py
"""Example 16.4 - plane-box: 8 corners in, only the ones within margin + gap come out.

plane_box (collision_primitive_core.py) returns the signed distance of all 8 box corners.
plane_box_wrapper calls write_contact once per corner with id_ = corner index, and write_contact
keeps a corner only if dist < margin + gap. So contact.geomcollisionid tells you WHICH corner.
A flat resting box gives 4 contacts, a box tilted about one edge gives 2, a box balanced on a
corner gives 1.
Run: python ex4_plane_box.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BOX_ON_PLANE = """
<mujoco model="plane_box">
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="box" pos="0 0 0.5">
      <freejoint/>
      <geom name="box" type="box" size="0.1 0.1 0.1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

HALF = 0.1
DEPTH = 0.001          # the lowest corner penetrates the floor by 1 mm in every world


def quat_axis_angle(axis, deg):
    axis = np.asarray(axis, float) / np.linalg.norm(axis)
    a = np.radians(deg) / 2
    return np.concatenate([[np.cos(a)], np.sin(a) * axis])


def corners_local():
    # same corner numbering as plane_box: bit 0 -> x, bit 1 -> y, bit 2 -> z
    return np.array([[HALF if i & 1 else -HALF, HALF if i & 2 else -HALF,
                      HALF if i & 4 else -HALF] for i in range(8)])


CASES = [("flat", [1, 0, 0], 0.0),
         ("tilt 10 deg about x (edge)", [1, 0, 0], 10.0),
         ("tilt 50 deg about (1,-1,0) (corner)", [1, -1, 0], 50.0)]
NWORLD = len(CASES)

mjm = mujoco.MjModel.from_xml_string(BOX_ON_PLANE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)

qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float64)
for w, (_, axis, deg) in enumerate(CASES):
    q = quat_axis_angle(axis, deg)
    rot = np.zeros(9)
    mujoco.mju_quat2Mat(rot, q)
    zc = (corners_local() @ rot.reshape(3, 3).T)[:, 2]
    qpos[w, 2] = -zc.min() - DEPTH        # lift so the lowest corner is 1 mm deep
    qpos[w, 3:7] = q
d.qpos.assign(qpos.astype(np.float32))
mjw.kinematics(m, d)
mjw.collision(m, d)

n = min(int(d.nacon.numpy()[0]), d.naconmax)
wid = d.contact.worldid.numpy()[:n]
gcid = d.contact.geomcollisionid.numpy()[:n]
dist = d.contact.dist.numpy()[:n]
xpos = d.geom_xpos.numpy()
xmat = d.geom_xmat.numpy()

out = []
for w, (name, _, _) in enumerate(CASES):
    # what plane_box computes for all 8 corners (plane normal +z through the origin)
    corner_z = (corners_local() @ xmat[w, 1].T + xpos[w, 1])[:, 2]
    sel = np.where(wid == w)[0]
    sel = sel[np.argsort(gcid[sel])]
    print(f"w{w} {name}")
    print("    corner dist (all 8):", np.round(corner_z, 4))
    print(f"    contacts written: {len(sel)}  geomcollisionid = {gcid[sel].tolist()}"
          f"  dist = {dist[sel]}")
    out.append({"name": name, "corner_dist": [round(float(v), 4) for v in corner_z],
                "gcid": gcid[sel].tolist()})
print("rule: corner i is kept when dist_i < margin + gap (= 0 here); 'resting' needs dist < 0")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"worlds": out}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 8. Box-box: separating axes and clipping
Default box-box is CONVEX (GJK/EPA + a max-area quad, 4 contacts). With NATIVECCD disabled, `box_box` (SAT over 15 axes + face clipping) keeps up to 8 clipped vertices, like MuJoCo C.

In [ ]:
# Example 16.7 - ex7_box_box_native.py
"""Example 16.7 - box-box: CONVEX (GJK/EPA + multi-contact) by default, PRIMITIVE without NATIVECCD.

MJ_COLLISION_TABLE sends BOX-BOX to the convex path. Setting DisableBit.NATIVECCD makes
collision_driver._narrowphase flip that one cell to PRIMITIVE, so box_box_wrapper (a port of
MuJoCo C's mjc_BoxBox: separating-axis test + face clipping) handles it instead. We stack two
boxes in three poses (aligned, top box yawed 20 and 45 degrees) and count box-box contacts:
MJWarp default, MJWarp with NATIVECCD disabled, and MuJoCo C (mj_forward) for reference.
Run: python ex7_box_box_native.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.io import override_model      # private API

np.set_printoptions(precision=4, suppress=True)


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


YAWS = [0.0, 20.0, 45.0]           # yaw of the top box, one world each
NWORLD = len(YAWS)

mjm = mujoco.MjModel.from_xml_string(box_stack(2))
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
for w, yaw in enumerate(YAWS):
    qpos[w, 2] = 0.0995                       # bottom box 0.5 mm into the floor
    qpos[w, 9] = 0.2985                       # top box 1 mm into the bottom box
    a = np.radians(yaw) / 2
    qpos[w, 10:14] = [np.cos(a), 0, 0, np.sin(a)]


def count_box_box(geom):
    return int(np.sum((geom[:, 0] == 1) & (geom[:, 1] == 2)))


results = {}
for label, overrides in [("MJWarp default", []),
                         ("MJWarp NATIVECCD off", ["opt.disableflags = nativeccd"])]:
    m = mjw.put_model(mjm)
    if overrides:
        override_model(m, overrides)           # mutates m in place, returns None
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
    d.qpos.assign(qpos.astype(np.float32))
    mjw.kinematics(m, d)
    mjw.collision(m, d)
    n = min(int(d.nacon.numpy()[0]), d.naconmax)
    wid = d.contact.worldid.numpy()[:n]
    geom = d.contact.geom.numpy()[:n]
    results[label] = [(count_box_box(geom[wid == w]), int(np.sum(wid == w))) for w in range(NWORLD)]
    print(f"{label:22s} disableflags = {int(m.opt.disableflags):6d}  overflow:",
          "none" if not d.overflow.numpy().any() else d.overflow.numpy())

for label, flag in [("MuJoCo C default", 0), ("MuJoCo C NATIVECCD off", 1)]:
    mjm_c = mujoco.MjModel.from_xml_string(box_stack(2))
    if flag:
        mjm_c.opt.disableflags |= mujoco.mjtDisableBit.mjDSBL_NATIVECCD
    mjd = mujoco.MjData(mjm_c)
    res = []
    for w in range(NWORLD):
        mjd.qpos[:] = qpos[w]
        mujoco.mj_forward(mjm_c, mjd)
        geom = np.array([c.geom for c in mjd.contact[:mjd.ncon]])
        res.append((count_box_box(geom), int(mjd.ncon)))
    results[label] = res

print("\nbox-box contacts (total incl. 4 floor contacts) per pose of the top box:")
print("                          " + "".join(f"yaw {y:4.0f}    " for y in YAWS))
for label, res in results.items():
    print(f"{label:24s}  " + "".join(f"{bb:2d} ({tot:2d})    " for bb, tot in res))

## 9. Surface velocity: conveyor belts
`<geom surfacevel=...>` is a batched field; `_contact_surfacevel` projects it on the contact frame and `_add_surface_vel` adds it to the friction rows. One belt speed per world. Both kernels run only if `m.flg_surfacevel` is True, which `put_model` copies from the MJCF: per-world values on a model whose XML has no nonzero surfacevel are silently ignored (set `m.flg_surfacevel = True` in that case).

In [ ]:
# Example 16.9 - ex9_conveyor.py
"""Example 16.9 - a conveyor belt with surface velocity, one belt speed per world.

The floor geom has surfacevel = (vx, vy, vz, wx, wy, wz) in its local frame. After the
narrowphase, collision_driver._contact_surfacevel turns it into contact.surfacevel
(tangent1, tangent2, spin components in the contact frame), and constraint._add_surface_vel adds
it to efc.Jqvel of the friction rows, so friction drags the box toward the belt speed. The
surface itself does not move. geom_surfacevel is a batched field, so every world gets its own
belt speed with put_model(batch_sizes=...). This only works because the XML already gives the
belt a nonzero surfacevel: m.flg_surfacevel is fixed at put_model, and with it False both
surface-velocity kernels are skipped, so per-world values would be silently ignored.
Run: python ex9_conveyor.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

CONVEYOR = """
<mujoco model="conveyor">
  <option timestep="0.002"/>
  <worldbody>
    <geom name="belt" type="plane" size="5 5 .1" surfacevel=".5 0 0  0 0 0"/>
    <body name="box" pos="0 0 0.1">
      <freejoint/>
      <geom name="box" type="box" size="0.1 0.1 0.1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

SPEEDS = np.array([0.25, 0.5, 1.0, -0.5], dtype=np.float32)   # belt vx per world
NWORLD = len(SPEEDS)
NSTEP = 300

mjm = mujoco.MjModel.from_xml_string(CONVEYOR)
m = mjw.put_model(mjm, batch_sizes={"geom_surfacevel": NWORLD})
sv = m.geom_surfacevel.numpy()                      # (nworld, ngeom) of vec6
sv[:, 0, 0] = SPEEDS
m.geom_surfacevel.assign(sv)
print("m.flg_surfacevel:", m.flg_surfacevel, "| geom_surfacevel shape:", m.geom_surfacevel.shape)

d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=48)
frames = []
for i in range(NSTEP):
    mjw.step(m, d)
    if (i + 1) % 50 == 0:
        print(f"step {i + 1:3d}  box vx per world:", d.qvel.numpy()[:, 0])
    if args.dump_json and i % 5 == 0:                # decimated (host reads: demo only)
        q = d.qpos.numpy()
        frames.append([[round(float(q[w, 0]), 3), round(float(d.qvel.numpy()[w, 0]), 3)]
                       for w in range(NWORLD)])
wp.synchronize()

print("belt speed per world:  ", SPEEDS)
print("final vx, 8 digits:    ", [f"{v:.8f}" for v in d.qvel.numpy()[:, 0]])
n = min(int(d.nacon.numpy()[0]), d.naconmax)
wid = d.contact.worldid.numpy()[:n]
svel = d.contact.surfacevel.numpy()[:n]
frame = d.contact.frame.numpy()[:n]
for w in range(NWORLD):
    i = np.where(wid == w)[0][0]
    print(f"w{w} contact.surfacevel {svel[i]}  tangent1 {frame[i, 1]}  tangent2 {frame[i, 2]}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "dt": 5 * float(mjm.opt.timestep),
                   "speeds": SPEEDS.tolist(), "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 10. Calling a core collider from your own kernel (private API)
The timing line is your device's number; on a CPU it is a CPU number, never a GPU number.

In [ ]:
# Example 16.10 - ex10_core_kernel.py
"""Example 16.10 - call MJWarp's pure sphere_sphere collider from your own Warp kernel.

collision_primitive_core.py holds the geometry of every primitive collider as plain @wp.func
functions with no Model/Data arguments. They are private API (they may change), but they are
ordinary Warp functions, so any kernel can call them. Here one thread tests one random sphere
pair; we check the result against NumPy and time the kernel.
Run: python ex10_core_kernel.py
"""
import time

import mujoco  # noqa: F401  (course import order)
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw  # noqa: F401
from mujoco_warp._src.collision_primitive_core import sphere_sphere   # private API

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NPAIR = 1_000_000 if ON_GPU else 100_000
NREP = 20


@wp.kernel
def sphere_pairs(p1: wp.array(dtype=wp.vec3), r1: wp.array(dtype=float),
                 p2: wp.array(dtype=wp.vec3), r2: wp.array(dtype=float),
                 dist_out: wp.array(dtype=float), pos_out: wp.array(dtype=wp.vec3),
                 normal_out: wp.array(dtype=wp.vec3), ntouch: wp.array(dtype=int)):
    i = wp.tid()
    dist, pos, n = sphere_sphere(p1[i], r1[i], p2[i], r2[i])
    dist_out[i] = dist
    pos_out[i] = pos
    normal_out[i] = n
    if dist < 0.0:
        wp.atomic_add(ntouch, 0, 1)          # count overlapping pairs


rng = np.random.default_rng(0)
P1 = rng.uniform(-1, 1, (NPAIR, 3)).astype(np.float32)
P2 = (P1 + rng.normal(0, 0.15, (NPAIR, 3))).astype(np.float32)
R1 = rng.uniform(0.02, 0.1, NPAIR).astype(np.float32)
R2 = rng.uniform(0.02, 0.1, NPAIR).astype(np.float32)

p1, p2 = wp.array(P1, dtype=wp.vec3), wp.array(P2, dtype=wp.vec3)
r1, r2 = wp.array(R1, dtype=float), wp.array(R2, dtype=float)
dist = wp.empty(NPAIR, dtype=float)
pos = wp.empty(NPAIR, dtype=wp.vec3)
normal = wp.empty(NPAIR, dtype=wp.vec3)
ntouch = wp.zeros(1, dtype=int)
inputs = [p1, r1, p2, r2]
outputs = [dist, pos, normal, ntouch]

wp.launch(sphere_pairs, dim=NPAIR, inputs=inputs, outputs=outputs)     # JIT compile + warm-up
wp.synchronize()
t0 = time.perf_counter()
for _ in range(NREP):
    ntouch.zero_()
    wp.launch(sphere_pairs, dim=NPAIR, inputs=inputs, outputs=outputs)
wp.synchronize()
dt = (time.perf_counter() - t0) / NREP

# NumPy reference: dist = |p2 - p1| - (r1 + r2), pos = p1 + n (r1 + dist / 2)
v = P2 - P1
L = np.linalg.norm(v, axis=1)
n_ref = v / L[:, None]
d_ref = L - (R1 + R2)
pos_ref = P1 + n_ref * (R1 + 0.5 * d_ref)[:, None]

print(f"device: {wp.get_device()}  pairs: {NPAIR}")
print("first 3 dist  :", dist.numpy()[:3])
print("NumPy  dist   :", d_ref[:3])
print("first pos     :", pos.numpy()[0], " normal:", normal.numpy()[0])
print(f"max |dist err| = {np.abs(dist.numpy() - d_ref).max():.2e}, "
      f"max |pos err| = {np.abs(pos.numpy() - pos_ref).max():.2e}")
print(f"overlapping pairs: {ntouch.numpy()[0]} (NumPy: {int((d_ref < 0).sum())})")
label = "GPU" if ON_GPU else "CPU (not a GPU number)"
print(f"{label}: {dt * 1e3:.2f} ms per launch, {NPAIR / dt / 1e6:.1f} M pairs/s")

## 11. Checking against MuJoCo C
Match contacts by geom pair and position, never by pool index.

In [ ]:
# Example 16.11 - ex11_vs_mujoco_c.py
"""Example 16.11 - every primitive collider in one scene, contact by contact against MuJoCo C.

The scene exercises plane-sphere, plane-capsule, plane-box, plane-cylinder, plane-ellipsoid,
sphere-capsule, sphere-box and capsule-box. We run mujoco.mj_forward (C, float64) and
mjw.kinematics + mjw.collision (MJWarp, float32) on the same qpos, match contacts by geom pair
(then by nearest position inside a pair), and print counts and the largest differences in
dist, pos and normal. Contact ORDER is not compared: MJWarp's pool order is arbitrary on a GPU.
Run: python ex11_vs_mujoco_c.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

MIXED = """
<mujoco model="primitive_zoo">
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 0.099"><freejoint/>
      <geom name="ball" type="sphere" size="0.1"/></body>
    <body name="rod" pos="0.5 0 0.049"><freejoint/>
      <geom name="rod" type="capsule" fromto="-0.15 0 0  0.15 0 0" size="0.05"/></body>
    <body name="pebble" pos="0.5 0.099 0.049"><freejoint/>
      <geom name="pebble" type="sphere" size="0.05"/></body>
    <body name="crate" pos="1 0 0.0995"><freejoint/>
      <geom name="crate" type="box" size="0.1 0.1 0.1"/></body>
    <body name="marble" pos="1 0 0.249"><freejoint/>
      <geom name="marble" type="sphere" size="0.05"/></body>
    <body name="post" pos="1.149 0 0.25"><freejoint/>
      <geom name="post" type="capsule" size="0.05 0.1"/></body>
    <body name="drum" pos="1.6 0 0.0995"><freejoint/>
      <geom name="drum" type="cylinder" size="0.06 0.1"/></body>
    <body name="egg" pos="2.0 0 0.0495"><freejoint/>
      <geom name="egg" type="ellipsoid" size="0.1 0.07 0.05"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(MIXED)
mjd = mujoco.MjData(mjm)
mujoco.mj_forward(mjm, mjd)
c_con = {}
for c in mjd.contact[:mjd.ncon]:
    key = tuple(sorted((int(c.geom[0]), int(c.geom[1]))))
    c_con.setdefault(key, []).append((c.dist, np.array(c.pos), np.array(c.frame[:3]),
                                      int(c.geom[0])))

m = mjw.put_model(mjm)
d = mjw.put_data(mjm, mjd, nworld=1, nconmax=48, njmax=192)
mjw.kinematics(m, d)
mjw.collision(m, d)
n = min(int(d.nacon.numpy()[0]), d.naconmax)
w_con = {}
for g, dist, pos, fr in zip(d.contact.geom.numpy()[:n], d.contact.dist.numpy()[:n],
                            d.contact.pos.numpy()[:n], d.contact.frame.numpy()[:n]):
    key = tuple(sorted((int(g[0]), int(g[1]))))
    w_con.setdefault(key, []).append((dist, pos, fr[0], int(g[0])))

name = lambda g: mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, g)
gtype = lambda g: mjw.GeomType(int(mjm.geom_type[g])).name.lower()
err = {"dist": 0.0, "pos": 0.0, "normal": 0.0}
print(f"C ncon = {mjd.ncon}, MJWarp nacon = {n}")
print("pair             types               C  MJWarp  normals agree")
for key in sorted(set(c_con) | set(w_con)):
    cs, ws = c_con.get(key, []), w_con.get(key, [])
    flips = 0
    for dist, pos, nrm, g0 in ws:                       # match each MJWarp contact to C
        j = int(np.argmin([np.linalg.norm(pos - cp) for _, cp, _, _ in cs]))
        cd, cp, cn, cg0 = cs[j]
        sign = 1.0 if g0 == cg0 else -1.0               # C may list the pair the other way
        flips += sign < 0
        err["dist"] = max(err["dist"], abs(dist - cd))
        err["pos"] = max(err["pos"], np.abs(pos - cp).max())
        err["normal"] = max(err["normal"], np.abs(sign * nrm - cn).max())
    a, b = key
    agree = "yes" if flips == 0 else f"{flips} listed reversed"
    print(f"{name(a):>6s}-{name(b):<8s}  {gtype(a):>8s}-{gtype(b):<9s} {len(cs):2d}  {len(ws):3d}"
          f"     {agree}")
print("max |dist diff|   = {:.2e} m".format(err["dist"]))
print("max |pos diff|    = {:.2e} m".format(err["pos"]))
print("max |normal diff| = {:.2e}".format(err["normal"]))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## Exercise 16.1: your own plane_capsule
Fill in the kernel: each end sphere of the capsule is a `plane_sphere`. The asserts compare with the contacts MJWarp wrote.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

TILTED_ROD = """
<mujoco><worldbody>
  <geom name="floor" type="plane" size="5 5 .1"/>
  <body name="rod" pos="0 0 0.3"><freejoint/>
    <geom name="rod" type="capsule" size="0.05 0.2" margin="1"/></body>
</worldbody></mujoco>"""
NWORLD = 6

@wp.kernel
def my_plane_capsule(geom_xpos: wp.array2d(dtype=wp.vec3), geom_xmat: wp.array2d(dtype=wp.mat33),
                     radius: float, half: float, dist_out: wp.array2d(dtype=float)):
    w = wp.tid()
    # TODO: plane normal = z-axis of geom 0; capsule axis = z-axis of geom 1
    # TODO: dist_out[w, 0] and dist_out[w, 1] for the two end spheres

mjm = mujoco.MjModel.from_xml_string(TILTED_ROD)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
rng = np.random.default_rng(3)
q = np.tile(mjm.qpos0, (NWORLD, 1))
q[:, 2] = rng.uniform(0.1, 0.4, NWORLD)
quat = rng.normal(size=(NWORLD, 4)); q[:, 3:7] = quat / np.linalg.norm(quat, axis=1, keepdims=True)
d.qpos.assign(q.astype(np.float32))
mjw.kinematics(m, d)
mjw.collision(m, d)

mine = wp.zeros((NWORLD, 2), dtype=float)
wp.launch(my_plane_capsule, dim=NWORLD, inputs=[d.geom_xpos, d.geom_xmat, 0.05, 0.2], outputs=[mine])
n = min(int(d.nacon.numpy()[0]), d.naconmax)
wid, dist = d.contact.worldid.numpy()[:n], d.contact.dist.numpy()[:n]
for w in range(NWORLD):
    theirs = np.sort(dist[wid == w])
    assert len(theirs) == 2, theirs
    assert np.allclose(np.sort(mine.numpy()[w]), theirs, atol=1e-5), (mine.numpy()[w], theirs)
print("my plane_capsule matches MJWarp in all", NWORLD, "worlds")

In [ ]:
#@title Exercise 16.1: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

TILTED_ROD = """
<mujoco><worldbody>
  <geom name="floor" type="plane" size="5 5 .1"/>
  <body name="rod" pos="0 0 0.3"><freejoint/>
    <geom name="rod" type="capsule" size="0.05 0.2" margin="1"/></body>
</worldbody></mujoco>"""
NWORLD = 6

@wp.kernel
def my_plane_capsule(geom_xpos: wp.array2d(dtype=wp.vec3), geom_xmat: wp.array2d(dtype=wp.mat33),
                     radius: float, half: float, dist_out: wp.array2d(dtype=float)):
    w = wp.tid()
    p = geom_xpos[w, 0]
    n = wp.vec3(geom_xmat[w, 0][0, 2], geom_xmat[w, 0][1, 2], geom_xmat[w, 0][2, 2])  # plane z-axis
    c = geom_xpos[w, 1]
    R = geom_xmat[w, 1]
    axis = wp.vec3(R[0, 2], R[1, 2], R[2, 2])                    # capsule axis = its local z
    dist_out[w, 0] = wp.dot(c + axis * half - p, n) - radius    # end sphere +axis
    dist_out[w, 1] = wp.dot(c - axis * half - p, n) - radius    # end sphere -axis

mjm = mujoco.MjModel.from_xml_string(TILTED_ROD)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
rng = np.random.default_rng(3)
q = np.tile(mjm.qpos0, (NWORLD, 1))
q[:, 2] = rng.uniform(0.1, 0.4, NWORLD)
quat = rng.normal(size=(NWORLD, 4)); q[:, 3:7] = quat / np.linalg.norm(quat, axis=1, keepdims=True)
d.qpos.assign(q.astype(np.float32))
mjw.kinematics(m, d)
mjw.collision(m, d)

mine = wp.zeros((NWORLD, 2), dtype=float)
wp.launch(my_plane_capsule, dim=NWORLD, inputs=[d.geom_xpos, d.geom_xmat, 0.05, 0.2], outputs=[mine])

n = min(int(d.nacon.numpy()[0]), d.naconmax)
wid, dist = d.contact.worldid.numpy()[:n], d.contact.dist.numpy()[:n]
for w in range(NWORLD):
    theirs = np.sort(dist[wid == w])
    assert len(theirs) == 2, theirs
    assert np.allclose(np.sort(mine.numpy()[w]), theirs, atol=1e-5), (mine.numpy()[w], theirs)
print("my plane_capsule matches MJWarp in all", NWORLD, "worlds")

## Exercise 16.2: predict the mixed friction
Fill in `predicted` with the 5-vector `contact.friction` for the floor against each ball. MJCF `friction="0.2"` sets only the first entry.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PAIRS = """
<mujoco><worldbody>
  <geom name="floor" type="plane" size="5 5 .1" friction="0.8 0.01 0.001"/>
  <body pos="0 0 .099"><freejoint/><geom name="g1" type="sphere" size=".1" friction="0.3"/></body>
  <body pos=".5 0 .099"><freejoint/><geom name="g2" type="sphere" size=".1" friction="1.2 0.002 0.002"/></body>
  <body pos="1 0 .099"><freejoint/><geom name="g3" type="sphere" size=".1" friction="0.2" priority="2"/></body>
  <body pos="1.5 0 .099"><freejoint/><geom name="g4" type="sphere" size=".1" friction="0.9 0.05" solmix="10"/></body>
  <body pos="2 0 .099"><freejoint/><geom name="g5" type="sphere" size=".1" friction="0.4 0 0" priority="1"/></body>
</worldbody></mujoco>"""

predicted = {"g1": [...], "g2": [...], "g3": [...], "g4": [...], "g5": [...]}   # your answers

mjm = mujoco.MjModel.from_xml_string(PAIRS)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=8, njmax=32)
mjw.kinematics(m, d)
mjw.collision(m, d)
n = min(int(d.nacon.numpy()[0]), d.naconmax)
for g, fr in zip(d.contact.geom.numpy()[:n], d.contact.friction.numpy()[:n]):
    nm = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, int(g[1]))
    assert np.allclose(fr, predicted[nm], rtol=1e-5, atol=1e-7), (nm, fr)
print("all 5 friction predictions correct")

In [ ]:
#@title Exercise 16.2: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PAIRS = """
<mujoco><worldbody>
  <geom name="floor" type="plane" size="5 5 .1" friction="0.8 0.01 0.001"/>
  <body pos="0 0 .099"><freejoint/><geom name="g1" type="sphere" size=".1" friction="0.3"/></body>
  <body pos=".5 0 .099"><freejoint/><geom name="g2" type="sphere" size=".1" friction="1.2 0.002 0.002"/></body>
  <body pos="1 0 .099"><freejoint/><geom name="g3" type="sphere" size=".1" friction="0.2" priority="2"/></body>
  <body pos="1.5 0 .099"><freejoint/><geom name="g4" type="sphere" size=".1" friction="0.9 0.05" solmix="10"/></body>
  <body pos="2 0 .099"><freejoint/><geom name="g5" type="sphere" size=".1" friction="0.4 0 0" priority="1"/></body>
</worldbody></mujoco>"""

# your prediction: the 5-vector contact.friction for floor + g1 .. g5
predicted = {
    "g1": [0.8, 0.8, 0.01, 0.001, 0.001],      # equal priority: element-wise max
    "g2": [1.2, 1.2, 0.01, 0.002, 0.002],      # max per element, mixes sources
    "g3": [0.2, 0.2, 0.005, 0.0001, 0.0001],   # priority 2 wins: g3's own (defaults)
    "g4": [0.9, 0.9, 0.05, 0.001, 0.001],      # solmix does not affect friction
    "g5": [0.4, 0.4, 1e-5, 1e-5, 1e-5],       # priority wins, zeros clamp to mjMINMU
}

mjm = mujoco.MjModel.from_xml_string(PAIRS)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=8, njmax=32)
mjw.kinematics(m, d)
mjw.collision(m, d)
n = min(int(d.nacon.numpy()[0]), d.naconmax)
for g, fr in zip(d.contact.geom.numpy()[:n], d.contact.friction.numpy()[:n]):
    nm = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_GEOM, int(g[1]))
    assert np.allclose(fr, predicted[nm], rtol=1e-5, atol=1e-7), (nm, fr)
print("all 5 friction predictions correct")

## Stretch: implement sphere_box
Transform the sphere centre into the box frame, clamp to the half-extents, handle the centre-inside case with the nearest face, and compare `dist` and the normal with MJWarp for 16 random poses.

In [ ]:
# your sphere_box here

In [ ]:
#@title Stretch: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

SPHERE_BOX = """
<mujoco><worldbody>
  <body name="box" pos="0 0 0"><freejoint/><geom name="box" type="box" size=".2 .1 .05" margin="1"/></body>
  <body name="ball" pos="0 0 .3"><freejoint/><geom name="ball" type="sphere" size=".08"/></body>
</worldbody></mujoco>"""

def my_sphere_box(c, r, bpos, bmat, half):
    local = bmat.T @ (c - bpos)                       # sphere centre in the box frame
    clamped = np.clip(local, -half, half)             # closest point of the box
    v = clamped - local
    L = np.linalg.norm(v)
    if L < 1e-15:                                     # centre inside the box: nearest face
        face = np.argmin(half - np.abs(local))
        nrm = np.zeros(3); nrm[face] = -np.sign(local[face])
        return -(half[face] - abs(local[face])) - r, bmat @ nrm
    return L - r, bmat @ (v / L)                      # normal points sphere -> box

NW = 16
mjm = mujoco.MjModel.from_xml_string(SPHERE_BOX)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NW, nconmax=4, njmax=16)
rng = np.random.default_rng(0)
q = np.tile(mjm.qpos0, (NW, 1))
quat = rng.normal(size=(NW, 4)); q[:, 3:7] = quat / np.linalg.norm(quat, axis=1, keepdims=True)
q[:, 7:10] = rng.uniform(-0.3, 0.3, (NW, 3))
d.qpos.assign(q.astype(np.float32))
mjw.kinematics(m, d); mjw.collision(m, d)
n = min(int(d.nacon.numpy()[0]), d.naconmax)
wid = d.contact.worldid.numpy()[:n]; dist = d.contact.dist.numpy()[:n]; nrm = d.contact.frame.numpy()[:n, 0]
xp, xm = d.geom_xpos.numpy(), d.geom_xmat.numpy()
for i in range(n):
    w = wid[i]
    dd, nn = my_sphere_box(xp[w, 1], 0.08, xp[w, 0], xm[w, 0], np.array([.2, .1, .05]))
    assert abs(dd - dist[i]) < 1e-5 and np.allclose(nn, nrm[i], atol=1e-4), (w, dd, dist[i], nn, nrm[i])
print("sphere_box matches on", n, "contacts")